**New benchmark analysis (2026-10-05).** Uses the available 4L/2H/K16 checkpoints with weight decay **0.1**, seeds 100–104. This is a new exploratory analysis, not reproduction of SEL-4L wd0.01. References/conflict definitions are frozen historical inputs, verified by hash. All model-dependent outputs are recomputed. Validation match splits support the stated comparisons; prior test access prevents a fresh confirmatory claim. Population and intervention scope apply to each reported result. See `config/analysis_plan.json` and `SURGICAL_FIXES.md`.

# MB03 — What the Model Does on Conflict Events

**Track B · Mechanistic interpretability of SEL-4L · Identity × Context project**

| Item | Content |
|---|---|
| **Question** | When the identity reference and the context reference disagree, which of the two actions does SEL-4L prefer, by how much, and how does that compare with what actually happened? |
| **Experiments** | E3 (which side the model picks; includes main test C1), E4 (do the five seeds agree?), sensitivity to τ, check by B5 agreement, model compared with the references |
| **Inputs** | MB01 conflict set and reference probabilities; MB02 groups, controls and pre-registration (all checked by fingerprint); SEL-4L seeds 100–104 |
| **Outputs** | `results/MB03_event_scores.parquet`, `results/MB03_summary.json`, `results/MB03_*.csv`, `figures/MB03_*` |
| **Test data** | Not used |

This notebook only looks at the model's inputs and outputs; nothing inside the model is examined yet. It describes the behaviour that the later notebooks try to explain.

**Background.** A conflict event is a validation event where a small identity-only model and a small context-only model (both trained in MB01) predict different top actions and clearly disagree. **a_id** is the identity model's top action, **a_ctx** the context model's top action.

**Calibration and population scope.** Mean eventwise log-odds and aggregate class-frequency log-odds are different statistics; their gap alone does not establish over-confidence. Corrected Section R compares probabilities and realised frequencies at the same aggregation level. Conflict rows were selected using two frozen reference predictors, partly for confident context predictions, so C1 applies to these selected events. Historical numerical examples and wd0.01 results are archived; all full-model values below are newly computed for wd0.1.

In [1]:
import os
import sys
import json
import time
import importlib
import importlib.util
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")


def find_root(start):
    """`start` (or one of its ancestors) holding both project folders directly, else None."""
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "model_training_evaluation").is_dir() and (candidate / "mi_analysis").is_dir():
            return candidate
    return None


def search_root(bases, max_depth=6):
    """Breadth-first search under each base (e.g. a mounted Drive) for the project root."""
    for base in bases:
        base = Path(base)
        if not base.is_dir():
            continue
        found = find_root(base)
        if found is not None:
            return found
        frontier = [(base, 0)]
        while frontier:
            folder, depth = frontier.pop(0)
            if depth >= max_depth:
                continue
            try:
                children = [c for c in folder.iterdir() if c.is_dir() and not c.name.startswith(".")]
            except PermissionError:
                continue
            for child in children:
                if child.name == "model_training_evaluation" and (child.parent / "mi_analysis").is_dir():
                    return child.parent
                frontier.append((child, depth + 1))
    return None


if "SOCCER_MI_ROOT" in os.environ:
    ROOT = Path(os.environ["SOCCER_MI_ROOT"])
elif IN_COLAB:
    ROOT = search_root(["/content", "/content/drive/MyDrive", "/content/drive/My Drive"])
    if ROOT is None:
        raise RuntimeError(
            "BLOCKED: could not find a folder holding both model_training_evaluation/ and mi_analysis/ "
            "under /content or the mounted Drive. Upload or sync the project there, or set "
            "SOCCER_MI_ROOT to its exact path in a cell ABOVE this one, for example: "
            "import os; os.environ['SOCCER_MI_ROOT'] = '/content/drive/MyDrive/.../soccer_mi'")
else:
    ROOT = find_root(Path.cwd())
    if ROOT is None:
        raise RuntimeError(f"BLOCKED: no ancestor of {Path.cwd()} holds both model_training_evaluation/ "
                           "and mi_analysis/. Run this notebook from inside the project, or set SOCCER_MI_ROOT.")
TRACK_A_DIR = ROOT / "model_training_evaluation"
MI_DIR = Path(os.environ.get("MI_ANALYSIS_DIR", ROOT / "mi_analysis")).resolve()
assert (TRACK_A_DIR / "src" / "data.py").exists(), (
    f"BLOCKED: {TRACK_A_DIR / 'src' / 'data.py'} not found -- ROOT resolved to {ROOT}, which is wrong. "
    "Set SOCCER_MI_ROOT explicitly.")
assert (TRACK_A_DIR / "artifacts" / "events_prepared.parquet").exists(), (
    f"BLOCKED: {TRACK_A_DIR / 'artifacts' / 'events_prepared.parquet'} not found -- ROOT is wrong.")
RUN_MODE = os.environ.get("MI_RUN_MODE", "full")
assert RUN_MODE in ("full", "smoke")

for folder in [MI_DIR, MI_DIR / "src", MI_DIR / "notebooks", MI_DIR / "tests"]:
    folder.mkdir(parents=True, exist_ok=True)
(MI_DIR / "src" / "__init__.py").touch()
os.chdir(MI_DIR)

sys.path[:] = [entry for entry in sys.path if entry not in ("", ".", str(MI_DIR))]
if str(TRACK_A_DIR) not in sys.path:
    sys.path.insert(0, str(TRACK_A_DIR))
spec = importlib.util.spec_from_file_location("mi_src", MI_DIR / "src" / "__init__.py",
                                              submodule_search_locations=[str(MI_DIR / "src")])
mi_src = importlib.util.module_from_spec(spec)
sys.modules["mi_src"] = mi_src
spec.loader.exec_module(mi_src)

print("Colab:", IN_COLAB, "| RUN_MODE:", RUN_MODE)
print("ROOT:", ROOT)
print("Track A (read only):", TRACK_A_DIR, "| exists:", TRACK_A_DIR.exists())
print("Track B:", MI_DIR)

Colab: False | RUN_MODE: full
ROOT: /home/aayush/code/soccer_mi
Track A (read only): /home/aayush/code/soccer_mi/model_training_evaluation | exists: True
Track B: /home/aayush/code/soccer_mi/.mi_analysis_rerun


In [2]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import src.data as data
import src.metrics as metrics
import src.training as training
import mi_src.paths as paths
import mi_src.fingerprint as fingerprint
import mi_src.mi_setup as mi_setup
import mi_src.hooks as hooks
import mi_src.mi_metrics as mi_metrics
for module in [data, metrics, training, paths, fingerprint, mi_setup, hooks, mi_metrics]:
    importlib.reload(module)

if getattr(paths, "SCHEMA_VERSION", 1) < 2:
    raise RuntimeError("BLOCKED: mi_analysis/src/paths.py is older than this notebook expects. "
                       "Re-run MB01 section 0 (its three %%writefile cells), then restart the kernel.")
paths.check_modules(**{name: module for name, module in
                       [("paths", paths), ("fingerprint", fingerprint), ("mi_setup", mi_setup),
                        ("patching", globals().get("patching")), ("circuits", globals().get("circuits"))]
                       if module is not None})

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

OUT = paths.output_dir(MI_DIR, RUN_MODE)
SETTINGS = paths.RUN_SETTINGS[RUN_MODE]
TRACK = paths.track_a_paths(TRACK_A_DIR)
MANIFEST = OUT / "manifests" / "sha_manifest.json"
REGISTRY = OUT / "manifests" / "experiment_registry.jsonl"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(int(os.environ.get("MI_CPU_THREADS", "4")))
if RUN_MODE == "full" and DEVICE != "cuda":
    print("Note: no GPU found. The full run works on CPU but is much slower.")
training.configure_determinism()
training.set_seed(0)
ACTIONS = list(data.ACTION_NAMES)
print("Device:", DEVICE, "| outputs:", OUT)
print("Settings:", SETTINGS)

Note: no GPU found. The full run works on CPU but is much slower.


Device: cpu | outputs: /home/aayush/code/soccer_mi/.mi_analysis_rerun
Settings: {'reference_epochs': 25, 'batch_windows': 256, 'n_bootstrap': 2000, 'n_draws': 5, 'n_permutations': 1000, 'n_random_circuits': 100, 'probe_train_events': 20000, 'max_events': 100000, 'probe_seeds': 3, 'scrub_draws': 5}


In [3]:
verified = fingerprint.verify_files(MANIFEST)
prereg = mi_setup.load_json(OUT / "config" / "preregistration.json")
mi_config = mi_setup.load_json(OUT / "config" / "mi_config.json")
IDENTITY = mi_config["identity_variable"]
PRIMARY_SEED = int(prereg["model"]["primary_seed"])


def run_dir(seed):
    """Run directory of one SEL-4L seed, re-rooted onto this checkout."""
    return paths.rebase_run_dir(prereg["model"]["run_dirs"][str(seed)], TRACK["root"])

SEEDS = [int(seed) for seed in prereg["model"]["run_dirs"]]
N_BOOT = SETTINGS["n_bootstrap"]
BATCH = SETTINGS["batch_windows"]

events = pd.read_parquet(TRACK["events"])
bundle = mi_setup.build_bundle(events, TRACK, DEVICE)
val_view = bundle["views"]["val"]
targets = bundle["targets"]
POP = OUT / "artifacts" / "populations"
val_conflicts = pd.read_parquet(POP / "val_conflicts.parquet")
controls = pd.read_parquet(POP / "matched_controls.parquet")
MODEL = mi_setup.load_model(run_dir(PRIMARY_SEED), bundle)
N_LAYERS = len(MODEL.blocks)
N_HEADS = MODEL.blocks[0].attention.n_heads
print(f"verified files: {len(verified)} | primary seed {PRIMARY_SEED} | identity variable: {IDENTITY}")
print(val_conflicts.groupby(["population", "half"]).size().unstack().to_string())

/home/aayush/code/soccer_mi/model_training_evaluation/src/batching.py:19: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  self.tensors[name] = torch.as_tensor(np.asarray(values), device=device)


verified files: 43 | primary seed 103 | identity variable: player
half        discovery  estimation
population                       
P-player         1215         529
index-0          4296        4651


## 1. Terms

| Term | Meaning here |
|---|---|
| **Logit** | The model's raw score for an action, before it is turned into a probability. Higher means more likely. |
| **Δ** | logit(a_ctx) − logit(a_id) at the event's scoring position. This equals log P(a_ctx) − log P(a_id). Δ > 0: the model gives more probability to the context action. Δ < 0: more to the identity action. Measured in nats; Δ = 1 means a_ctx is about 2.7 times as likely as a_id. |
| **Which side the model picks** | Whether the model gives more probability to a_ctx or to a_id. |
| **Realised action** | The action that actually happened. |
| **Data log-odds** (empirical log-odds) | log[(n(realised = a_ctx) + 0.5) / (n(realised = a_id) + 0.5)] over a group of events: how strongly the real data favour a_ctx over a_id. ~~If the model's probabilities are well matched to the data, its mean Δ should be close to this.~~ *[Corrected 2026-10-04 (F10): mean Δ is a mean of per-event log-odds and this is a log-odds of aggregate counts. They differ even for a calibrated model. A compatible comparison is in Section R.]* |
| **Δ adjusted for action frequency** (class-prior adjustment) | Δ − [log π(a_ctx) − log π(a_id)], where π is the share of each action in the training data. It is the part of Δ that is not explained by one action simply being more common overall. |

## 2. Scoring every validation conflict event with all five checkpoints

For each event and each seed we record: the 15 logits at the scoring position, P(a_ctx), P(a_id), the model's top action and its probability, and Δ. The matched non-conflict controls are scored with their partner's action pair, so their Δ compares the same two actions.

In [4]:
def run_dir(seed):
    """Run directory of one SEL-4L seed, re-rooted onto this checkout."""
    return paths.rebase_run_dir(prereg["model"]["run_dirs"][str(seed)], TRACK["root"])


run_dirs = {int(seed): run_dir(seed) for seed in SEEDS}
models = {seed: (MODEL if seed == PRIMARY_SEED else mi_setup.load_model(run_dirs[seed], bundle)) for seed in SEEDS}

@torch.no_grad()
def score(model, table):
    rows = []
    for part, inputs, positions, a_ctx, a_id in mi_setup.event_batches(bundle, val_view, table, BATCH):
        logits = mi_metrics.at_target(model(inputs), positions)
        log_probabilities = torch.log_softmax(logits, dim=-1)
        index = torch.arange(len(positions), device=logits.device)
        rows.append(pd.DataFrame({
            "delta": mi_metrics.delta(logits, a_ctx, a_id),
            "log_p_ctx": log_probabilities[index, a_ctx].cpu().numpy(),
            "log_p_id": log_probabilities[index, a_id].cpu().numpy(),
            "model_top": logits.argmax(-1).cpu().numpy(),
            "p_top": log_probabilities.max(-1).values.exp().cpu().numpy(),
        }))
    return pd.concat(rows, ignore_index=True)

scores = {}
for seed in SEEDS:
    scores[seed] = score(models[seed], val_conflicts)
event_scores = val_conflicts.copy()
for column in ["delta", "log_p_ctx", "log_p_id", "model_top", "p_top"]:
    event_scores[column] = scores[PRIMARY_SEED][column].to_numpy()
for seed in SEEDS:
    event_scores[f"delta_seed{seed}"] = scores[seed]["delta"].to_numpy()
control_scores = controls.copy()
if len(controls):
    control_scores["delta"] = score(MODEL, controls)["delta"].to_numpy()
event_scores.to_parquet(OUT / "results" / "MB03_event_scores.parquet", index=False)
control_scores.to_parquet(OUT / "results" / "MB03_control_scores.parquet", index=False)
print(f"scored {len(event_scores):,} conflict events x {len(SEEDS)} seeds and {len(control_scores):,} controls")

scored 10,691 conflict events x 5 seeds and 252 controls


Historical result prose removed for the new model; use the freshly executed outputs above.


In [5]:
train_prior = np.bincount(targets[events["split"].to_numpy() == "train"], minlength=15) / (events["split"] == "train").sum()
event_scores["prior_log_ratio"] = np.log(train_prior[event_scores["a_ctx"]]) - np.log(train_prior[event_scores["a_id"]])
event_scores["delta_prior_adjusted"] = event_scores["delta"] - event_scores["prior_log_ratio"]

def empirical_log_odds(frame):
    n_ctx = (frame["action_id"] == frame["a_ctx"]).sum()
    n_id = (frame["action_id"] == frame["a_id"]).sum()
    return float(np.log((n_ctx + 0.5) / (n_id + 0.5)))

def describe(name, frame):
    summary = mi_metrics.summarise(frame["delta"], frame["match_id"], N_BOOT)
    adjusted = mi_metrics.summarise(frame["delta_prior_adjusted"], frame["match_id"], N_BOOT)
    return {"subset": name, "n": summary["n"], "matches": summary["n_matches"], "mean delta": summary["mean"],
            "CI low": summary["ci_low"], "CI high": summary["ci_high"], "share delta > 0": float((frame["delta"] > 0).mean()),
            "empirical log-odds": empirical_log_odds(frame), "prior-adjusted delta": adjusted["mean"],
            "model top = a_ctx": float((frame["model_top"] == frame["a_ctx"]).mean()),
            "model top = a_id": float((frame["model_top"] == frame["a_id"]).mean()),
            "model top = realised": float((frame["model_top"] == frame["action_id"]).mean()),
            "realised = a_ctx": float((frame["action_id"] == frame["a_ctx"]).mean()),
            "realised = a_id": float((frame["action_id"] == frame["a_id"]).mean())}

e = event_scores
subsets = [("P-all", e), ("P-all, estimation half (C1)", e[e.half == "estimation"]), ("P-all, discovery half", e[e.half == "discovery"]),
           ("P-player", e[e.population == "P-player"]), ("index-0", e[e.population == "index-0"]),
           ("genuine", e[e.genuine]), ("coupled", e[~e.genuine]),
           ("B5 agrees with a_ctx", e[e.b5_agrees_with_a_ctx]), ("B5 disagrees", e[~e.b5_agrees_with_a_ctx])]
if "a_id_margin_band" in e:
    for band in ["< 0.02", "0.02 to 0.05", "> 0.05"]:
        subsets.append((f"a_id margin {band}", e[e.a_id_margin_band == band]))
behaviour = pd.DataFrame([describe(name, frame) for name, frame in subsets if len(frame)])
behaviour.to_csv(OUT / "results" / "MB03_behaviour.csv", index=False)
print(behaviour.round(3).to_string(index=False))

c1 = behaviour[behaviour.subset == "P-all, estimation half (C1)"].iloc[0]
print(f"\nC1 (estimation half): mean delta {c1['mean delta']:.4f}, 95% CI [{c1['CI low']:.4f}, {c1['CI high']:.4f}], n = {int(c1['n'])}")

                     subset     n  matches  mean delta  CI low  CI high  share delta > 0  empirical log-odds  prior-adjusted delta  model top = a_ctx  model top = a_id  model top = realised  realised = a_ctx  realised = a_id
                      P-all 10691       35       6.250   6.130    6.367            0.983               3.788                 7.962              0.931             0.006                 0.887             0.874            0.020
P-all, estimation half (C1)  5180       17       6.209   6.077    6.338            0.981               3.853                 7.919              0.928             0.007                 0.889             0.878            0.019
      P-all, discovery half  5511       18       6.288   6.101    6.499            0.985               3.726                 8.003              0.933             0.004                 0.885             0.870            0.021
                   P-player  1744       27       5.963   5.728    6.187            0.982            

In [6]:
print("Conflict events against their matched non-conflict controls (P-player, same action pair)")
if len(control_scores):
    paired = control_scores.merge(event_scores[["event_row", "delta", "match_id"]], left_on="partner_event_row",
                                  right_on="event_row", suffixes=("_control", "_conflict"))
    difference = paired["delta_conflict"] - paired["delta_control"]
    paired_summary = mi_metrics.summarise(difference, paired["match_id_conflict"], N_BOOT)
    print(f"pairs: {len(paired)} | mean delta, conflict {paired['delta_conflict'].mean():.3f} vs control "
          f"{paired['delta_control'].mean():.3f} | difference {paired_summary['mean']:.3f} "
          f"[{paired_summary['ci_low']:.3f}, {paired_summary['ci_high']:.3f}]")
    for tier in sorted(paired["match_tier"].unique()):
        part = paired[paired.match_tier == tier]
        print(f"  tier {tier}: {len(part)} pairs, mean difference {(part['delta_conflict'] - part['delta_control']).mean():.3f}")
else:
    paired_summary = None
    print("no matched controls available")

Conflict events against their matched non-conflict controls (P-player, same action pair)
pairs: 252 | mean delta, conflict 4.319 vs control 2.422 | difference 1.897 [1.525, 2.266]
  tier 1: 161 pairs, mean difference 1.784
  tier 2: 91 pairs, mean difference 2.097


In [7]:
pair_rows = []
for (a_id, a_ctx), frame in event_scores.groupby(["a_id", "a_ctx"]):
    if len(frame) < 30:
        continue
    summary = mi_metrics.summarise(frame["delta"], frame["match_id"], N_BOOT)
    pair_rows.append({"a_id": ACTIONS[a_id], "a_ctx": ACTIONS[a_ctx], "n": len(frame), "mean delta": summary["mean"],
                      "CI low": summary["ci_low"], "CI high": summary["ci_high"],
                      "empirical log-odds": empirical_log_odds(frame),
                      "prior log-ratio": float(np.log(train_prior[a_ctx]) - np.log(train_prior[a_id]))})
pairs = pd.DataFrame(pair_rows).sort_values("n", ascending=False)
pairs.to_csv(OUT / "results" / "MB03_pairs.csv", index=False)
print(pairs.round(3).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
axes[0].hist(event_scores["delta"], bins=60, alpha=0.6, density=True, label="conflict events (P-all)")
if len(control_scores):
    axes[0].hist(control_scores["delta"], bins=60, alpha=0.6, density=True, label="matched non-conflict controls")
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set_xlabel("delta = logit(a_ctx) - logit(a_id) (nats)"); axes[0].set_ylabel("density"); axes[0].legend()
axes[0].set_title("Arbitration on conflict events and on controls")
if len(pairs):
    axes[1].errorbar(pairs["empirical log-odds"], pairs["mean delta"],
                     yerr=[pairs["mean delta"] - pairs["CI low"], pairs["CI high"] - pairs["mean delta"]], fmt="o")
    limit = float(np.nanmax(np.abs(np.concatenate([pairs["empirical log-odds"], pairs["mean delta"]])))) + 0.5
    axes[1].plot([-limit, limit], [-limit, limit], "--", color="grey", label="model = data")
    axes[1].set_xlabel("empirical log-odds of a_ctx over a_id in the data"); axes[1].set_ylabel("mean delta (model)")
    axes[1].set_title("Per (a_id, a_ctx) pair, n >= 30"); axes[1].legend()
plt.tight_layout(); plt.savefig(OUT / "figures" / "MB03_delta.png", bbox_inches="tight"); plt.show()

      a_id          a_ctx    n  mean delta  CI low  CI high  empirical log-odds  prior log-ratio
PASS_SHORT           DUEL 1878       6.967   6.798    7.147               5.762           -2.023
PASS_SHORT          CARRY 1797       6.599   6.386    6.787               8.183           -0.319
PASS_SHORT   INTERCEPTION 1276       4.964   4.797    5.127               3.343           -2.212
PASS_SHORT       FOUL_WON  740      10.984  10.857   11.099               6.199           -3.076
 PASS_LONG          CARRY  553       7.155   7.007    7.290               5.905            0.083
 PASS_LONG      GK_ACTION  545       7.319   7.172    7.448               6.974           -2.515
PASS_SHORT      CLEARANCE  538       2.959   2.706    3.177               2.799           -2.532
PASS_SHORT FOUL_COMMITTED  490       6.368   6.183    6.526               4.556           -3.026
 PASS_LONG           DUEL  352       7.914   7.658    8.123               5.386           -1.621
PASS_SHORT           SHOT  340

/tmp/ipykernel_300487/2454527332.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUT / "figures" / "MB03_delta.png", bbox_inches="tight"); plt.show()


Historical result prose removed for the new model; use the freshly executed outputs above.


**Researcher interpretation** *(written by the researcher after reading the numbers above; the code never writes these words)*

- What was tested:
- Why it was tested:
- What was observed:
- What the result supports:
- What it contradicts:
- Alternative explanations that remain:
- What it does not establish:
- Evidence label (Observed / Supported / Strong causal evidence / Hypothesized):

## 4. E4 — Is this behaviour the same across training seeds?

The per-event Δ of the five checkpoints is compared in pairs with Spearman's rank correlation (1 = the events are ranked in exactly the same order). Each correlation gets an interval from resampling matches. The mean Δ of each seed is also reported.

In [8]:
agreement_rows = []
match_array = event_scores["match_id"].to_numpy()
unique_matches = np.unique(match_array)
rng = np.random.default_rng(0)
resamples = [rng.choice(unique_matches, size=len(unique_matches), replace=True) for _ in range(min(N_BOOT, 500))]
row_lists = {m: np.where(match_array == m)[0] for m in unique_matches}
for i, seed_a in enumerate(SEEDS):
    for seed_b in SEEDS[i + 1:]:
        a = event_scores[f"delta_seed{seed_a}"].to_numpy()
        b = event_scores[f"delta_seed{seed_b}"].to_numpy()
        draws = []
        for chosen in resamples:
            rows = np.concatenate([row_lists[m] for m in chosen])
            draws.append(mi_metrics.spearman(a[rows], b[rows]))
        agreement_rows.append({"seed a": seed_a, "seed b": seed_b, "spearman": mi_metrics.spearman(a, b),
                               "CI low": np.percentile(draws, 2.5), "CI high": np.percentile(draws, 97.5)})
agreement = pd.DataFrame(agreement_rows)
agreement.to_csv(OUT / "results" / "MB03_seed_agreement.csv", index=False)
print(agreement.round(3).to_string(index=False))
seed_means = {seed: float(event_scores[f"delta_seed{seed}"].mean()) for seed in SEEDS}
print("\nmean delta by seed:", {seed: round(value, 4) for seed, value in seed_means.items()},
      f"| SD across seeds {np.std(list(seed_means.values()), ddof=1):.4f}")

 seed a  seed b  spearman  CI low  CI high
    100     101     0.892   0.881    0.902
    100     102     0.871   0.860    0.881
    100     103     0.875   0.864    0.887
    100     104     0.901   0.894    0.908
    101     102     0.884   0.871    0.897
    101     103     0.894   0.885    0.904
    101     104     0.897   0.885    0.907
    102     103     0.876   0.862    0.887
    102     104     0.884   0.872    0.894
    103     104     0.868   0.856    0.879

mean delta by seed: {100: 6.4196, 101: 6.1614, 102: 6.4627, 103: 6.2499, 104: 6.6494} | SD across seeds 0.1906


Historical result prose removed for the new model; use the freshly executed outputs above.


In [9]:
tau_record = mi_setup.load_json(OUT / "artifacts" / "conflict" / "tau_selection.json")
sensitivity_rows = []
for row in tau_record["sweep"]:
    if row["tau"] < tau_record["tau"]:
        continue
    subset = event_scores[event_scores["JSD"] > row["tau"] - 1e-9]
    if len(subset) == 0:
        continue
    summary = mi_metrics.summarise(subset["delta"], subset["match_id"], N_BOOT)
    sensitivity_rows.append({"percentile": row["percentile"], "tau": row["tau"], "n": summary["n"],
                             "mean delta": summary["mean"], "CI low": summary["ci_low"], "CI high": summary["ci_high"],
                             "share delta > 0": float((subset["delta"] > 0).mean()),
                             "realised = a_ctx": float((subset["action_id"] == subset["a_ctx"]).mean())})
sensitivity = pd.DataFrame(sensitivity_rows)
print(sensitivity.round(4).to_string(index=False))
fig, axis = plt.subplots(figsize=(7, 3.5))
axis.errorbar(sensitivity["percentile"], sensitivity["mean delta"],
              yerr=[sensitivity["mean delta"] - sensitivity["CI low"], sensitivity["CI high"] - sensitivity["mean delta"]], fmt="o-")
axis.set_xlabel("tau percentile (MB01 sweep)"); axis.set_ylabel("mean delta (nats)"); axis.set_title("Arbitration as the JSD cut-off rises")
plt.tight_layout(); plt.savefig(OUT / "figures" / "MB03_tau_sensitivity.png", bbox_inches="tight"); plt.show()

 percentile    tau     n  mean delta  CI low  CI high  share delta > 0  realised = a_ctx
         75 0.3471 10691      6.2499  6.1298   6.3668           0.9833            0.8739
         80 0.3813  8553      6.5344  6.4120   6.6583           0.9843            0.8893
         85 0.4330  6415      6.9012  6.7566   7.0546           0.9875            0.9005
         90 0.5267  4277      7.7401  7.5913   7.8842           0.9937            0.9462
         95 0.5838  2139      8.7839  8.6273   8.9359           0.9939            0.9710


/tmp/ipykernel_300487/2775285691.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUT / "figures" / "MB03_tau_sensitivity.png", bbox_inches="tight"); plt.show()


Historical result prose removed for the new model; use the freshly executed outputs above.


In [10]:
agreement_class = np.where(event_scores["model_top"] == event_scores["a_ctx"], "a_ctx",
                           np.where(event_scores["model_top"] == event_scores["a_id"], "a_id", "neither"))
print(pd.Series(agreement_class).value_counts(normalize=True).round(3).to_string())
neither = event_scores[agreement_class == "neither"]
top_cases = neither.groupby(["a_id", "a_ctx", "model_top"]).size().sort_values(ascending=False).head(5)
for (a_id, a_ctx, top), count in top_cases.items():
    frame = neither[(neither.a_id == a_id) & (neither.a_ctx == a_ctx) & (neither.model_top == top)]
    realised = frame["action_id"].value_counts(normalize=True).head(3)
    print(f"a_id {ACTIONS[a_id]:10s} a_ctx {ACTIONS[a_ctx]:13s} model {ACTIONS[top]:13s} n={count:4d} | realised: "
          + ", ".join(f"{ACTIONS[k]} {v:.2f}" for k, v in realised.items()))

a_ctx      0.931
neither    0.064
a_id       0.006
a_id PASS_LONG  a_ctx CLEARANCE     model GK_ACTION     n=  69 | realised: GK_ACTION 0.67, RECOVERY 0.28, INTERCEPTION 0.03
a_id PASS_LONG  a_ctx CLEARANCE     model RECOVERY      n=  46 | realised: RECOVERY 0.54, GK_ACTION 0.33, CLEARANCE 0.07
a_id PASS_SHORT a_ctx FOUL_COMMITTED model DUEL          n=  45 | realised: FOUL_COMMITTED 0.51, DUEL 0.40, INTERCEPTION 0.07
a_id PASS_SHORT a_ctx DUEL          model FOUL_COMMITTED n=  41 | realised: FOUL_COMMITTED 0.66, DUEL 0.32, CLEARANCE 0.02
a_id PASS_SHORT a_ctx RECOVERY      model CLEARANCE     n=  39 | realised: CLEARANCE 0.72, RECOVERY 0.13, PASS_SHORT 0.05


In [11]:
summary = {"C1": {"n": int(c1["n"]), "mean_delta": float(c1["mean delta"]), "ci_low": float(c1["CI low"]),
                  "ci_high": float(c1["CI high"])},
           "behaviour": behaviour.to_dict(orient="records"),
           "paired_conflict_minus_control": paired_summary,
           "seed_agreement": agreement.to_dict(orient="records"), "seed_means": seed_means,
           "tau_sensitivity": sensitivity.to_dict(orient="records"), "run_mode": RUN_MODE}
mi_setup.save_json(OUT / "results" / "MB03_summary.json", summary)
fingerprint.record_files(MANIFEST, {"results/MB03_event_scores.parquet": OUT / "results" / "MB03_event_scores.parquet",
                                    "results/MB03_summary.json": OUT / "results" / "MB03_summary.json"})
fingerprint.register(REGISTRY, {"notebook": "MB03", "experiment": "E3/E4 behaviour", "population": "P-all val",
                                "n": len(event_scores), "C1_mean": float(c1["mean delta"]), "seed": PRIMARY_SEED,
                                "run_mode": RUN_MODE})
print("MB03 results saved.")

MB03 results saved.


Historical result prose removed for the new model; use the freshly executed outputs above.


## R. Calibration compared at the same aggregation level (REPAIR 2026-10-04, F10)

For each subset of Section 3 and for each (a_id, a_ctx) pair with at least 30 events:

* **mean predicted p(a_ctx) vs realised share of a_ctx**, and the same for a_id. The per-event paired difference
  p − 1[realised] has a valid match-bootstrap interval. A calibrated model has a difference near 0 (calibration-in-the-large
  for that action on that population).
* **aggregate model log-odds** log(Σ p_ctx / Σ p_id) vs **data log-odds** log(n_ctx / n_id). Both are ratios of sums over the
  same events, so they are comparable. The difference gets a match bootstrap; no 0.5 smoothing is used.
* a **reliability table** for p(a_ctx) on P-all (deciles of predicted probability).

This uses only `MB03_event_scores` (primary seed 103) and runs no model.

In [12]:
import mi_src.mi_metrics as mi_metrics
importlib.reload(mi_metrics)
if getattr(mi_metrics, "REPAIR_VERSION", None) != "2026-10-04":
    raise RuntimeError("BLOCKED: mi_metrics.py on disk is not the repaired version; re-run MB02's repaired %%writefile cell and restart.")

def calibration_row(name, frame):
    p_ctx, p_id = np.exp(frame["log_p_ctx"].to_numpy()), np.exp(frame["log_p_id"].to_numpy())
    realised_ctx = (frame["action_id"] == frame["a_ctx"]).to_numpy().astype(float)
    realised_id = (frame["action_id"] == frame["a_id"]).to_numpy().astype(float)
    m = frame["match_id"].to_numpy()
    ctx = mi_metrics.probability_calibration(p_ctx, realised_ctx, m, N_BOOT)
    ide = mi_metrics.probability_calibration(p_id, realised_id, m, N_BOOT)
    agg = mi_metrics.aggregate_log_odds(p_ctx, p_id, realised_ctx, realised_id, m, N_BOOT)
    return {"subset": name, "n": len(frame), "mean p(a_ctx)": ctx["mean predicted"], "realised a_ctx": ctx["realised share"],
            "p(a_ctx) - realised": ctx["predicted minus realised"], "CI low (a_ctx)": ctx["CI low"], "CI high (a_ctx)": ctx["CI high"],
            "mean p(a_id)": ide["mean predicted"], "realised a_id": ide["realised share"],
            "p(a_id) - realised": ide["predicted minus realised"], "CI low (a_id)": ide["CI low"], "CI high (a_id)": ide["CI high"],
            "model aggregate log-odds": agg["model aggregate log-odds"], "data log-odds": agg["data log-odds"],
            "model - data (aggregate)": agg["model minus data"], "CI low (aggregate)": agg["CI low"], "CI high (aggregate)": agg["CI high"],
            "mean delta (for reference; not comparable)": float(frame["delta"].mean())}

calibration = pd.DataFrame([calibration_row(name, frame) for name, frame in subsets if len(frame)])
calibration.to_csv(OUT / "results" / "MB03_calibration_corrected.csv", index=False)
print(calibration.round(4).to_string(index=False))

pair_calibration = pd.DataFrame([{**calibration_row(f"{ACTIONS[a_id]} vs {ACTIONS[a_ctx]}", frame), "a_id": ACTIONS[a_id], "a_ctx": ACTIONS[a_ctx]}
                                 for (a_id, a_ctx), frame in event_scores.groupby(["a_id", "a_ctx"]) if len(frame) >= 30])
pair_calibration.to_csv(OUT / "results" / "MB03_pairs_calibration_corrected.csv", index=False)
reliability = mi_metrics.reliability_bins(np.exp(event_scores["log_p_ctx"]), (event_scores["action_id"] == event_scores["a_ctx"]).astype(float), 10)
reliability.to_csv(OUT / "results" / "MB03_reliability_corrected.csv", index=False)
print("\nreliability of p(a_ctx), P-all (deciles of predicted probability):")
print(reliability.round(4).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
finite = pair_calibration[np.isfinite(pair_calibration["data log-odds"])]
axes[0].errorbar(finite["data log-odds"], finite["model aggregate log-odds"],
                 yerr=[finite["model - data (aggregate)"] - finite["CI low (aggregate)"], finite["CI high (aggregate)"] - finite["model - data (aggregate)"]],
                 fmt="o", alpha=0.8)
limit = float(np.nanmax(np.abs(np.concatenate([finite["data log-odds"], finite["model aggregate log-odds"]])))) + 0.5
axes[0].plot([-limit, limit], [-limit, limit], "--", color="grey", label="model = data (calibrated)")
axes[0].set_xlabel("data log-odds log(n_ctx / n_id)"); axes[0].set_ylabel("model log(sum p_ctx / sum p_id)")
axes[0].set_title("Per (a_id, a_ctx) pair, n >= 30: comparable aggregates", fontsize=9); axes[0].legend(fontsize=8)
axes[1].plot(reliability["mean predicted"], reliability["realised share"], "o-")
axes[1].plot([0, 1], [0, 1], "--", color="grey"); axes[1].set_xlabel("mean predicted p(a_ctx) in bin"); axes[1].set_ylabel("realised share of a_ctx")
axes[1].set_title("Reliability of p(a_ctx), P-all", fontsize=9)
plt.tight_layout(); plt.savefig(OUT / "figures" / "MB03_calibration_corrected.png", bbox_inches="tight"); plt.show()
mi_setup.save_json(OUT / "results" / "MB03_summary_corrected.json", {"calibration": calibration.to_dict(orient="records"),
    "pairs": pair_calibration.to_dict(orient="records"), "reliability": reliability.to_dict(orient="records"), "run_mode": RUN_MODE})
fingerprint.record_files(MANIFEST, {"results/MB03_summary_corrected.json": OUT / "results" / "MB03_summary_corrected.json"})

                     subset     n  mean p(a_ctx)  realised a_ctx  p(a_ctx) - realised  CI low (a_ctx)  CI high (a_ctx)  mean p(a_id)  realised a_id  p(a_id) - realised  CI low (a_id)  CI high (a_id)  model aggregate log-odds  data log-odds  model - data (aggregate)  CI low (aggregate)  CI high (aggregate)  mean delta (for reference; not comparable)
                      P-all 10691         0.8630          0.8739              -0.0109         -0.0157          -0.0060        0.0197         0.0197             -0.0000        -0.0034          0.0032                    3.7804         3.7905                   -0.0101             -0.1901               0.1494                                      6.2499
P-all, estimation half (C1)  5180         0.8631          0.8780              -0.0149         -0.0222          -0.0077        0.0206         0.0185              0.0021        -0.0032          0.0071                    3.7342         3.8581                   -0.1239             -0.4350             

/tmp/ipykernel_300487/2525729459.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUT / "figures" / "MB03_calibration_corrected.png", bbox_inches="tight"); plt.show()


{'artifacts/conflict/conflict_set.parquet': {'path': 'artifacts/conflict/conflict_set.parquet',
  'bytes': 1735548,
  'sha256': '19d6197298f38803450ba3befbc0e1b744b6ab900eecb0f6374783c4cf2263b3',
  'recorded_at': '2026-10-05T17:22:56.743259+00:00'},
 'artifacts/conflict/identity_audit/id_mlp_128_64_seed0.pt': {'path': 'artifacts/conflict/identity_audit/id_mlp_128_64_seed0.pt',
  'bytes': 330757,
  'sha256': '590fbc131586f5297fcd9ddfcdf7f2e1c503cf13e9f584142a5c31a10a79339c',
  'recorded_at': '2026-10-05T17:22:56.743711+00:00'},
 'artifacts/conflict/identity_audit/id_mlp_512_512_256_seed0.pt': {'path': 'artifacts/conflict/identity_audit/id_mlp_512_512_256_seed0.pt',
  'bytes': 2280753,
  'sha256': 'eab81928ab7743ddbe2cbdb1dc60d5b584408386e896705115145e7c8a6d62a2',
  'recorded_at': '2026-10-05T17:22:56.746435+00:00'},
 'artifacts/conflict/predictions/test_probs.parquet': {'path': 'artifacts/conflict/predictions/test_probs.parquet',
  'bytes': 7982855,
  'sha256': 'd21c146acd76ee17cb82670b

**How to read Section R.** An interval for "p − realised" that excludes 0 means that, on that population, the model's average
probability for that action differs from how often the action happened. That is a calibration-in-the-large statement for that
action. The aggregate comparison does the same for the log-odds of the two candidate actions. Neither speaks to the
mechanism, and on conflict events both are conditional on how the conflicts were selected (F11).

**Researcher interpretation** *(to be written after reading the numbers)*

- What was observed:
- What it supports / contradicts:
- What it does not establish: